# A.1 Aim: Encoder-Decoder Architecture with Attention
- Implement encoder decoder architecture with attention on the given dataset
- Compare the performance of the above model with simple encoder-decoder architecture

# A.2 Prerequisites
Concept of Language Modeling, RNN, LSTM, Encoder-Decoder Architecture, Attention mechanism

# A.3 Learning outcomes
After completing the experiment, you will be able to:
- Design, implement and train Encoder-Decoder Architecture for the Machine Translation Application using attention model
- Appreciate the significance of attention mechanism

# A.4 Theory
### A.4.1 Introduction to Encoder-Decoder Architecture
- **Encoder-decoder architecture**
  - A type of neural network model used to transform one sequence into another
  - Popular in machine translation, text summarization, image captioning, and speech recognition

### A.4.2 Structure of an Encoder-Decoder
- **Encoder**: Accepts input sequence `x` and produces contextualized representations `h`
  (LSTMs, CNNs, and Transformers can all serve as encoders)
- **Context Vector**: A function of `h` that conveys the essence of the input to the decoder
- **Decoder**: Accepts `c` and generates an output sequence `y`
  (can be any sequence architecture: RNN, LSTM, GRU)

### A.4.3 Encoder-Decoder with Attention
Fig 1: Machine Translation using Encoder-Decoder Architecture with Attention  
$e_{ij} = a(s_{i-1}, h_j)$  
$\alpha_{ij} = \frac{\exp(e_{ij})}{\sum_{j=1}^M \exp(e_{ij})}$

# A.5 Task to be completed
- Design, implement and train an English-to-Hindi Machine Translation System using
  Encoder-Decoder Architecture with attention
- Compare results with the simple encoder-decoder model from Lab 6.
  Keep dataset size, batch size, number of epochs, and all other relevant parameters identical.


In [ ]:
# Installing and Importing necessary libraries
# !pip install datasets gradio
import random
import math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from datasets import load_dataset
import gradio as gr

# ── Vocabulary token indices ─────────────────────────────────────────────────
PAD_IDX = 0
SOS_IDX = 1
EOS_IDX = 2
UNK_IDX = 3

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# ── Hardware ─────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
# 1. Dataset Loading and Preprocessing
print("Loading a subset of cfilt/iitb-english-hindi dataset...")
dataset = load_dataset("cfilt/iitb-english-hindi", split="train[:5000]")


class Vocab:
    """Simple whitespace-tokenized vocabulary with fixed special tokens."""

    SPECIAL_TOKENS = {
        "<pad>": PAD_IDX,
        "<sos>": SOS_IDX,
        "<eos>": EOS_IDX,
        "<unk>": UNK_IDX,
    }

    def __init__(self, name: str):
        self.name = name
        self.word2index: dict[str, int] = dict(self.SPECIAL_TOKENS)
        self.index2word: dict[int, str] = {v: k for k, v in self.SPECIAL_TOKENS.items()}
        self.n_words = len(self.SPECIAL_TOKENS)

    def add_sentence(self, sentence: str):
        for word in sentence.split():
            self._add_word(word)

    def _add_word(self, word: str):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.index2word[self.n_words] = word
            self.n_words += 1


en_vocab = Vocab("english")
hi_vocab = Vocab("hindi")

en_sentences, hi_sentences = [], []
for ex in dataset:
    en_sent = ex["translation"]["en"].lower()
    hi_sent = ex["translation"]["hi"]
    en_vocab.add_sentence(en_sent)
    hi_vocab.add_sentence(hi_sent)
    en_sentences.append(en_sent)
    hi_sentences.append(hi_sent)

print(f"English Vocab Size: {en_vocab.n_words}")
print(f"Hindi Vocab Size:   {hi_vocab.n_words}")


def sentence_to_tensor(vocab: Vocab, sentence: str, max_len: int = 20) -> torch.Tensor:
    """Tokenize, truncate, add SOS/EOS, and pad a sentence into a fixed-length tensor."""
    indexes = [vocab.word2index.get(w, UNK_IDX) for w in sentence.split()]
    indexes = [SOS_IDX] + indexes[: max_len - 2] + [EOS_IDX]
    indexes += [PAD_IDX] * (max_len - len(indexes))
    return torch.tensor(indexes, dtype=torch.long, device=device)


from torch.utils.data import DataLoader, TensorDataset

MAX_LEN = 20
BATCH_SIZE = 64

src_tensors = torch.stack([sentence_to_tensor(en_vocab, s, MAX_LEN) for s in en_sentences])
trg_tensors = torch.stack([sentence_to_tensor(hi_vocab, s, MAX_LEN) for s in hi_sentences])

train_dataset = TensorDataset(src_tensors, trg_tensors)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)


In [ ]:
# 2. Simple Encoder-Decoder Architecture

class SimpleEncoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        _, (hidden, cell) = self.rnn(embedded)
        return hidden, cell


class SimpleDecoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout, batch_first=True)
        self.fc_out = nn.Linear(hid_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):
        embedded = self.dropout(self.embedding(input.unsqueeze(1)))
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(1))
        return prediction, hidden, cell


class SimpleSeq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size, trg_len = trg.shape
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size, device=self.device)
        hidden, cell = self.encoder(src)

        dec_input = trg[:, 0]
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(dec_input, hidden, cell)
            outputs[:, t] = output
            dec_input = trg[:, t] if random.random() < teacher_forcing_ratio else output.argmax(1)

        return outputs


In [ ]:
# 3. Encoder-Decoder with Attention

class Attention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        self.attn = nn.Linear(hid_dim * 3, hid_dim)
        self.v = nn.Linear(hid_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        src_len = encoder_outputs.shape[1]
        # Expand decoder hidden state across all source positions
        hidden_expanded = hidden.unsqueeze(1).repeat(1, src_len, 1)
        energy = torch.tanh(self.attn(torch.cat((hidden_expanded, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)
        return F.softmax(attention, dim=1)


class AttnEncoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(
            emb_dim, hid_dim, n_layers, dropout=dropout, batch_first=True, bidirectional=True
        )
        self.fc = nn.Linear(hid_dim * 2, hid_dim)
        self.dropout = nn.Dropout(dropout)
        self.n_layers = n_layers

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.rnn(embedded)

        # Merge bidirectional states: [fwd, bwd] → single vector per layer
        hidden = torch.tanh(self.fc(torch.cat((hidden[-2], hidden[-1]), dim=1)))
        cell = torch.tanh(self.fc(torch.cat((cell[-2], cell[-1]), dim=1)))

        hidden = hidden.unsqueeze(0).repeat(self.n_layers, 1, 1)
        cell = cell.unsqueeze(0).repeat(self.n_layers, 1, 1)
        return outputs, hidden, cell


class AttentionDecoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.attention = Attention(hid_dim)
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(
            hid_dim * 2 + emb_dim, hid_dim, n_layers, dropout=dropout, batch_first=True
        )
        self.fc_out = nn.Linear(hid_dim * 3 + emb_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell, encoder_outputs):
        embedded = self.dropout(self.embedding(input.unsqueeze(1)))  # (B, 1, emb)

        attn_weights = self.attention(hidden[-1], encoder_outputs).unsqueeze(1)  # (B, 1, src)
        weighted = torch.bmm(attn_weights, encoder_outputs)                       # (B, 1, hid)

        rnn_input = torch.cat((embedded, weighted), dim=2)
        output, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))

        # Squeeze sequence dim for FC layer
        prediction = self.fc_out(
            torch.cat((output.squeeze(1), weighted.squeeze(1), embedded.squeeze(1)), dim=1)
        )
        return prediction, hidden, cell


class AttnSeq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size, trg_len = trg.shape
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size, device=self.device)
        encoder_outputs, hidden, cell = self.encoder(src)

        dec_input = trg[:, 0]
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(dec_input, hidden, cell, encoder_outputs)
            outputs[:, t] = output
            dec_input = trg[:, t] if random.random() < teacher_forcing_ratio else output.argmax(1)

        return outputs


In [ ]:
# 4. Training

# ── Hyperparameters ───────────────────────────────────────────────────────────
INPUT_DIM  = en_vocab.n_words
OUTPUT_DIM = hi_vocab.n_words
ENC_EMB_DIM = DEC_EMB_DIM = 256
HID_DIM    = 512
N_LAYERS   = 2
DROPOUT    = 0.5
EPOCHS     = 5
CLIP       = 1.0

# ── Model instantiation ───────────────────────────────────────────────────────
simple_enc = SimpleEncoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM, N_LAYERS, DROPOUT)
simple_dec = SimpleDecoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, N_LAYERS, DROPOUT)
simple_model = SimpleSeq2Seq(simple_enc, simple_dec, device).to(device)

attn_enc = AttnEncoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM, N_LAYERS, DROPOUT)
attn_dec = AttentionDecoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, N_LAYERS, DROPOUT)
attn_model = AttnSeq2Seq(attn_enc, attn_dec, device).to(device)


def init_weights(model: nn.Module):
    for name, param in model.named_parameters():
        nn.init.normal_(param.data, mean=0, std=0.01) if "weight" in name else nn.init.constant_(param.data, 0)


simple_model.apply(init_weights)
attn_model.apply(init_weights)

optimizer_simple = optim.Adam(simple_model.parameters())
optimizer_attn   = optim.Adam(attn_model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)


def train_epoch(model, optimizer, dataloader):
    model.train()
    total_loss = 0.0
    for src, trg in dataloader:
        src, trg = src.to(device), trg.to(device)
        optimizer.zero_grad()
        output = model(src, trg)

        # Flatten output/target (skip <sos> token at position 0)
        output_dim = output.shape[-1]
        loss = criterion(output[:, 1:].reshape(-1, output_dim), trg[:, 1:].reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)


def train_model(model, optimizer, dataloader, model_name):
    print(f"\n--- Training {model_name} ---")
    for epoch in range(1, EPOCHS + 1):
        loss = train_epoch(model, optimizer, dataloader)
        print(f"Epoch {epoch:02}/{EPOCHS} | Loss: {loss:.3f}")


print("Starting Training...")
train_model(simple_model, optimizer_simple, train_loader, "Simple Encoder-Decoder")
train_model(attn_model,   optimizer_attn,   train_loader, "Encoder-Decoder with Attention")


In [ ]:
# 5. Inference & Gradio Dashboard

def translate(sentence: str, model, is_attn: bool = False) -> str:
    """Greedily decode a translated Hindi sentence from an English input."""
    model.eval()
    indexes = [SOS_IDX] + [en_vocab.word2index.get(w, UNK_IDX) for w in sentence.lower().split()] + [EOS_IDX]
    src_tensor = torch.tensor(indexes, dtype=torch.long, device=device).unsqueeze(0)

    with torch.no_grad():
        if is_attn:
            encoder_outputs, hidden, cell = model.encoder(src_tensor)
        else:
            hidden, cell = model.encoder(src_tensor)

    trg_indexes = [SOS_IDX]
    for _ in range(MAX_LEN):
        trg_tensor = torch.tensor([trg_indexes[-1]], dtype=torch.long, device=device)
        with torch.no_grad():
            if is_attn:
                output, hidden, cell = model.decoder(trg_tensor, hidden, cell, encoder_outputs)
            else:
                output, hidden, cell = model.decoder(trg_tensor, hidden, cell)

        pred_token = output.argmax(1).item()
        trg_indexes.append(pred_token)
        if pred_token == EOS_IDX:
            break

    tokens = [hi_vocab.index2word.get(i, "<unk>") for i in trg_indexes[1:-1]]
    return " ".join(tokens)


def translation_dashboard(text: str):
    return (
        translate(text, simple_model, is_attn=False),
        translate(text, attn_model,   is_attn=True),
    )


interface = gr.Interface(
    fn=translation_dashboard,
    inputs=gr.Textbox(lines=2, placeholder="Enter English text here...", label="Input English Sentence"),
    outputs=[
        gr.Textbox(label="Simple Encoder-Decoder Output"),
        gr.Textbox(label="Attention Encoder-Decoder Output"),
    ],
    title="English to Hindi Machine Translation",
    description="Comparing Simple Encoder-Decoder with Attention-based Encoder-Decoder.",
)

interface.launch(share=False)
